In [1]:
%pip install topf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 44.4 MB/s eta 0:00:00


In [2]:
import topf  # runs ~6min

## Synthetic Point Clouds

In [394]:
import numpy as np
import plotly.graph_objects as go

def noisy_circle(x0, y0, z0=None, n=200, r=1.0, noise=0.1, plane="xy"):
    theta = np.linspace(0, 2*np.pi, n)

    x = x0 + r * np.cos(theta) + np.random.normal(0, noise, n)
    y = y0 + r * np.sin(theta) + np.random.normal(0, noise, n)
    if z0 is None:
        return x, y

    z = z0 + np.random.normal(0, noise, n)

    if plane == "yz":
        return z, x, y
    if plane == "xz":
        return x, z, y

    return x, y, z

def plot_cloud(x, y, z=None):
    if z is None:
        fig = go.Figure(data=[go.Scatter(
            x=x, y=y,
            mode='markers',
            marker=dict(
                size=3
            )
        )])
        fig.update_xaxes(constrain='domain')
        fig.update_yaxes(scaleanchor="x", scaleratio=1)
    else:
        fig = go.Figure(data=[go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(
                size=2
            )
        )])

        fig.update_layout(
            scene=dict(
                aspectmode='data'
            ),
            margin=dict(l=0, r=0, t=0, b=0),
        )
    fig.show()

In [671]:
def make_audi_cloud_2d(x0=0, y0=0, z0=0, n=1000, r=1.3, noise=0.1):
    c1 = np.stack(noisy_circle(0, 0, n=n, r=r, noise=noise), axis=1)
    c2 = np.stack(noisy_circle(2, 0, n=n, r=r, noise=noise), axis=1)
    c3 = np.stack(noisy_circle(4, 0, n=n, r=r, noise=noise), axis=1)
    c4 = np.stack(noisy_circle(6, 0, n=n, r=r, noise=noise), axis=1)
    return np.vstack((c1, c2, c3, c4)) + np.array([x0, y0, z0])

def make_star_cloud_2d(x0=0, y0=0, z0=0, n=1000, r=1.3, noise=0.1):
    c1 = np.stack(noisy_circle(0, 0, n=n, r=r, noise=noise), axis=1)
    c2 = np.stack(noisy_circle(-np.sqrt(2*2-1**2), -1, n=n, r=r, noise=noise), axis=1)
    c3 = np.stack(noisy_circle(np.sqrt(2**2-1**2), -1, n=n, r=r, noise=noise), axis=1)
    c4 = np.stack(noisy_circle(0, 2, n=n, r=r, noise=noise), axis=1)
    return np.vstack((c1, c2, c3, c4)) + np.array([x0, y0, z0])

def make_bracelet_cloud_3d(x0=0, y0=0, z0=0, n=1000, r=1.3, inner_r=1, noise=0.1):
    c1 = np.stack(noisy_circle(0, 0, inner_r, n=n, r=r, noise=noise), axis=1)
    c2 = np.stack(noisy_circle(0, 0, -inner_r, n=n, r=r, noise=noise), axis=1)
    c3 = np.stack(noisy_circle(0, 0, inner_r, n=n, r=r, noise=noise, plane="xz"), axis=1)
    c4 = np.stack(noisy_circle(0, 0, -inner_r, n=n, r=r, noise=noise, plane="xz"), axis=1)
    return np.vstack((c1, c2, c3, c4)) + np.array([x0, y0, z0])

def make_one_point_cloud_3d(x0=0, y0=0, z0=0, n=1000, r=1.3, noise=0.1):
    c1 = np.stack(noisy_circle(0, r, 0, n=n, r=r, noise=noise), axis=1)
    c2 = np.stack(noisy_circle(0, -r, 0, n=n, r=r, noise=noise), axis=1)
    c3 = np.stack(noisy_circle(0, r, 0, n=n, r=r, noise=noise, plane="xz"), axis=1)
    c4 = np.stack(noisy_circle(0, -r, 0, n=n, r=r, noise=noise, plane="xz"), axis=1)
    return np.vstack((c1, c2, c3, c4))

def make_sphere_cloud_3d(x0=0, y0=0, z0=0, n=4000, r=2.6, noise=0.01):
    points = np.random.normal(0, 1, (n, 3))
    norms = np.linalg.norm(points, axis=1, keepdims=True)

    noise = np.random.normal(0, noise, (n, 3))
    return (points / norms) * r + np.array([x0, y0, z0]) + noise

In [562]:
audi_cloud_2d = make_audi_cloud_2d()
plot_cloud(audi_cloud_2d[:, 0], audi_cloud_2d[:, 1])

In [563]:
star_cloud_2d = make_star_cloud_2d()
plot_cloud(star_cloud_2d[:, 0], star_cloud_2d[:, 1])

In [598]:
bracelet_cloud_3d = make_bracelet_cloud_3d(0, 0, 0, r=1.3, inner_r=1)
plot_cloud(bracelet_cloud_3d[:, 0], bracelet_cloud_3d[:, 1], bracelet_cloud_3d[:, 2])

In [592]:
one_point_cloud_3d = make_one_point_cloud_3d(r=2.6)
plot_cloud(one_point_cloud_3d[:, 0], one_point_cloud_3d[:, 1], one_point_cloud_3d[:, 2])

In [673]:
two_sphere_one_ring_cloud_3d = np.vstack(
    (
        make_sphere_cloud_3d(-2, 0, 0),
        make_sphere_cloud_3d(2, 0, 0),
        np.stack(noisy_circle(-2, 0, 0, n=4000, r=4), axis=1)
    )
)
plot_cloud(
    two_sphere_one_ring_cloud_3d[:, 0],
    two_sphere_one_ring_cloud_3d[:, 1],
    two_sphere_one_ring_cloud_3d[:, 2]
)

## Topological Intersection Graph

In [600]:
_DIM_COLORS = {0: "cadetblue", 1: "chocolate", 2: "darkorchid"}
_DIM_LABELS = {0: "H0 component", 1: "H1 loop", 2: "H2 void"}


def intersection_graph(
    points, top_features, labels, feature_dim,
    edge_threshold=0.1, show_cloud=True,
    similarity="cosine",
    title="Topological Intersection Graph",
):
    """
    Visualises cluster-to-cluster topological intersection confidence.

    Parameters
    ----------
    points : np.ndarray (N, 2|3)
        Original point cloud.  Node positions are cluster centroids.
    top_features : np.ndarray (N, F)
        Topological features from topf(return_dict=True).
    labels : np.ndarray (N,)
        Cluster labels from topf(return_dict=True)["labels"].
    feature_dim : array-like (F,)
        Homology dimension of each feature column, from
        topf(return_dict=True)["feature_dim"].
    edge_threshold : float
        Minimum similarity to draw an edge.
    show_cloud : bool
        Overlay a faint point cloud for spatial reference.
    similarity : {"cosine", "dot"}
        "cosine"  — normalised dot product in [-1,1], clipped to >=0.
        "dot"     — raw dot product (useful when feature magnitudes matter).
    title : str

    Returns
    -------
    W : np.ndarray (K, K)
        Symmetric similarity matrix between cluster mean feature vectors
        (diagonal zero).
    """
    K = int(labels.max()) + 1
    F = top_features.shape[1]
    is3 = points.shape[1] == 3
    feature_dim = np.asarray(feature_dim)

    # per-cluster statistics
    centroids = np.array([points[labels == k].mean(0) for k in range(K)])
    sizes = np.array([(labels == k).sum() for k in range(K)])
    means = np.array([
        top_features[labels == k].mean(0) if (labels == k).any() else np.zeros(F)
        for k in range(K)
    ])

    # dominant homology dimension per cluster
    dominant = np.array([
        max(
            {d: means[k, feature_dim == d].sum() for d in np.unique(feature_dim)},
            key=lambda d: means[k, feature_dim == d].sum(),
        ) if F else 0
        for k in range(K)
    ], dtype=int)

    # similarity matrix
    if similarity == "cosine":
        normed = means / (np.linalg.norm(means, axis=1, keepdims=True) + 1e-12)
        W = np.clip(normed @ normed.T, 0, 1)
    else:  # dot
        W = means @ means.T

    np.fill_diagonal(W, 0.0)
    max_w = W.max() or 1.0

    fig = go.Figure()

    # background point cloud
    if show_cloud:
        cloud_marker = dict(
            size=1.5 if is3 else 3,
            color=labels,
            colorscale="Portland",
            # opacity=0.7,
        )
        scatter_cls = go.Scatter3d if is3 else go.Scatter
        coords = dict(x=points[:, 0], y=points[:, 1])
        if is3:
            coords["z"] = points[:, 2]
        fig.add_trace(scatter_cls(
            **coords, mode="markers",
            marker=cloud_marker, hoverinfo="skip", showlegend=False,
        ))

    # edges
    for i in range(K):
        for j in range(i + 1, K):
            w = W[i, j]
            if w < edge_threshold:
                continue

            width_rel = w / max_w
            color = f"rgba(128,128,128,1)"
            mid = (centroids[i] + centroids[j]) / 2
            ht = f"C{i} ↔ C{j}<br>sim={w:.3f}<extra></extra>"

            if is3:
                fig.add_trace(go.Scatter3d(
                    x=[centroids[i, 0], centroids[j, 0], None],
                    y=[centroids[i, 1], centroids[j, 1], None],
                    z=[centroids[i, 2], centroids[j, 2], None],
                    mode="lines",
                    line=dict(color=color, width=10*width_rel),
                    hovertemplate=ht, showlegend=False,
                ))
            else:
                fig.add_trace(go.Scatter(
                    x=[centroids[i, 0], centroids[j, 0]],
                    y=[centroids[i, 1], centroids[j, 1]],
                    mode="lines",
                    line=dict(color=color, width=5*width_rel),
                    hovertemplate=ht, showlegend=False,
                ))
                fig.add_annotation(
                    x=mid[0], y=mid[1], text=f"{w:.2f}",
                    showarrow=False,
                    font=dict(size=9, color="rgba(180,178,172,1)"),
                    bgcolor="rgba(23,22,20,0.85)", borderpad=2,
                )

    # nodes
    for k in range(K):
        mk = dict(
            size=40*np.sqrt(sizes[k] / sizes.max()),
            color=_DIM_COLORS[dominant[k]],
            line=dict(width=3 if is3 else 1.5)
        )
        ht = (
            f"<b>Cluster {k}</b><br>N={sizes[k]}<br>"
            f"Dominant: {_DIM_LABELS.get(dominant[k], f'H{dominant[k]}')}<br><extra></extra>"
        )

        if is3:
            fig.add_trace(go.Scatter3d(
                x=[centroids[k, 0]], y=[centroids[k, 1]], z=[centroids[k, 2]],
                mode="markers+text", marker=mk,
                text=[f"C{k}"], textposition="top center",
                hovertemplate=ht, showlegend=False,
            ))
        else:
            fig.add_trace(go.Scatter(
                x=[centroids[k, 0]], y=[centroids[k, 1]],
                mode="markers+text", marker=mk,
                text=[f"C{k}"], textposition="top center",
                hovertemplate=ht, showlegend=False,
            ))

    # H0, H1, H2 legend
    for i, (d, c) in enumerate(_DIM_COLORS.items()):
        fig.add_annotation(
            xref="paper", yref="paper", x=1.1, y=0.95 - i * 0.08,
            text=f"<span style='color:{c}'>● {_DIM_LABELS[d]}</span>",
            showarrow=False, align="left"
        )

    if is3:
        fig.update_layout(scene=dict(
            aspectmode="data",
            xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
        ))
    else:
        fig.update_xaxes(visible=False, constrain="domain")
        fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)

    fig.update_layout(
        title=dict(text=title),
         margin=dict(l=0, r=150, t=50, b=0)
    )
    fig.show()
    return W

In [568]:
features, topf_dict = topf.topf(audi_cloud_2d, return_dict=True)
similarity_matrix = intersection_graph(audi_cloud_2d, features, topf_dict["labels"], topf_dict["feature_dim"])

In [569]:
features, topf_dict = topf.topf(star_cloud_2d, return_dict=True)
similarity_matrix = intersection_graph(star_cloud_2d, features, topf_dict["labels"], topf_dict["feature_dim"])

In [606]:
features, topf_dict = topf.topf(bracelet_cloud_3d, max_hom_dim=2, return_dict=True)
similarity_matrix = intersection_graph(bracelet_cloud_3d, features, topf_dict["labels"], topf_dict["feature_dim"])

In [614]:
features, topf_dict = topf.topf(one_point_cloud_3d, return_dict=True)
similarity_matrix = intersection_graph(one_point_cloud_3d, features, topf_dict["labels"], topf_dict["feature_dim"])

In [680]:
features, topf_dict = topf.topf(two_sphere_one_ring_cloud_3d, max_hom_dim=2, max_rel_quot=0.5, max_total_quot=0.3, return_dict=True)
similarity_matrix = intersection_graph(two_sphere_one_ring_cloud_3d, features, topf_dict["labels"], topf_dict["feature_dim"])

In [681]:
similarity_matrix

array([[0.        , 0.87436282, 0.2905502 ],
       [0.87436282, 0.        , 0.07824524],
       [0.2905502 , 0.07824524, 0.        ]])

In [682]:
fig = go.Figure(data=go.Heatmap(
    z=similarity_matrix,
    text=similarity_matrix,
    texttemplate="%{text:.2f}",
    textfont={"size": 12},
    x=["C0", "C1", "C2"],
    y=["C0", "C1", "C2"],
))
fig.update_layout(title="Clusters Intersection Heatmap")
fig.show()